<a href="https://colab.research.google.com/github/PranjanaNarayan/fraud-detection-system/blob/main/rag_document_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community sentence-transformers faiss-cpu pypdf transformers torch -q
print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All packages installed!


In [4]:
from google.colab import files
import os

print("Please upload any PDF file...")
print("You can use:")
print("- Any textbook PDF")
print("- Any research paper")
print("- Any company report")
print("- Even your college notes PDF")

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"\nUploaded: {pdf_filename}")

Please upload any PDF file...
You can use:
- Any textbook PDF
- Any research paper
- Any company report
- Even your college notes PDF


Saving DA2026.pdf to DA2026.pdf

Uploaded: DA2026.pdf


In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# First install the missing package
import subprocess
subprocess.run(["pip", "install", "langchain-text-splitters", "-q"])

# Automatically finds your uploaded PDF
pdf_filename = None
for f in os.listdir('/content'):
    if f.endswith('.pdf'):
        pdf_filename = f
        print(f"Found your PDF: {pdf_filename}")
        break

if pdf_filename is None:
    print("No PDF found!")
else:
    print("Loading your PDF...")
    loader = PyPDFLoader(f"/content/{pdf_filename}")
    documents = loader.load()

    print(f"Total pages loaded: {len(documents)}")
    print(f"Sample text:")
    print(documents[0].page_content[:300])

    print("\nSplitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(chunks)}")
    print("Done!")

Found your PDF: attention_paper.pdf
Loading your PDF...
Total pages loaded: 15
Sample text:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par

Splitting into chunks...
Total chunks created: 93
Done!


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model...")
print("This takes 2-3 minutes on first run...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating vector database...")
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector database created!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

Loading embedding model...
This takes 2-3 minutes on first run...


/tmp/ipykernel_1924/2075906654.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating vector database...
Vector database created!
Total vectors stored: 93


In [10]:
from transformers import pipeline

print("Loading free LLM model...")
print("This takes 3-5 minutes first time...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM loaded successfully!")

Loading free LLM model...
This takes 3-5 minutes first time...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

LLM loaded successfully!


In [11]:
def ask_question(question, top_k=3):
    print(f"\nQuestion: {question}")
    print("-" * 50)

    relevant_docs = vectorstore.similarity_search(question, k=top_k)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f"""Answer the question based on the context below.
If the answer is not in the context, say I don't know.

Context:
{context}

Question: {question}

Answer:"""

    answer = generate_answer(prompt)

    print(f"Answer: {answer}")
    print(f"\nSources: {len(relevant_docs)} chunks from your PDF")
    for i, doc in enumerate(relevant_docs):
        page = doc.metadata.get('page', 0) + 1
        print(f"  Source {i+1}: Page {page}")

    return answer

print("RAG system is ready!")

RAG system is ready!


In [13]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

print("Loading FLAN-T5 model...")
print("This takes 3-5 minutes first time...")

model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

def generate_answer(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt",
                      max_length=512, truncation=True)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def ask_question(question, top_k=3):
    print(f"\nQuestion: {question}")
    print("-" * 50)
    relevant_docs = vectorstore.similarity_search(question, k=top_k)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    prompt = f"""Answer the question based on the context below.
If the answer is not in the context, say I don't know.

Context:
{context}

Question: {question}

Answer:"""
    answer = generate_answer(prompt)
    print(f"Answer: {answer}")
    print(f"\nSources: {len(relevant_docs)} chunks from your PDF")
    for i, doc in enumerate(relevant_docs):
        page = doc.metadata.get('page', 0) + 1
        print(f"  Source {i+1}: Page {page}")
    return answer

print("Model loaded and RAG system is ready!")

Loading FLAN-T5 model...
This takes 3-5 minutes first time...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded and RAG system is ready!


In [14]:
ask_question("What is the main topic of this document?")
ask_question("What are the key points mentioned?")
ask_question("Write a brief summary of this document")


Question: What is the main topic of this document?
--------------------------------------------------
Answer: Attention is All You Need

Sources: 3 chunks from your PDF
  Source 1: Page 1
  Source 2: Page 13
  Source 3: Page 7

Question: What are the key points mentioned?
--------------------------------------------------
Answer: reducing the attention key size dk hurts model quality

Sources: 3 chunks from your PDF
  Source 1: Page 9
  Source 2: Page 5
  Source 3: Page 13

Question: Write a brief summary of this document
--------------------------------------------------
Answer: [28] Romain Paulus, Caiming Xiong, and Richard Socher. Learning accurate, compact, and interpretable tree annotation. In Proceedings of the 21st International Conference on Computational Linguistics and 44th Annual Meeting of the ACL , pages 433–440. ACL, July 2006. arXiv:1607.06450, 2016. [2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly learning to align and trans

'[28] Romain Paulus, Caiming Xiong, and Richard Socher. Learning accurate, compact, and interpretable tree annotation. In Proceedings of the 21st International Conference on Computational Linguistics and 44th Annual Meeting of the ACL , pages 433–440. ACL, July 2006. arXiv:1607.06450, 2016. [2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly learning to align and translate. CoRR, abs/1409.0473, 2014. [3] Denny Britz, Anna Goldie, Minh-Thang Luong, and Quoc V . Le. Massive exploration of neural machine translation architectures. CoRR, abs/1409.0473, 2014. [4] Jianpeng Cheng, Li Dong, and Mir'